# Домашнее задание 3

В этом задании напишем простое решение классификации датасета `FashionMNIST`, а затем будем его улучшать с помощью:
- dropout;
- batch normalization;
- LR scheduler;

В конце сохраним модель в файл и убедимся, что этот файл можем в дальнейшем прочитать.

In [45]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torchvision.datasets import FashionMNIST
from torchvision.transforms import ToTensor
from dataclasses import dataclass

In [46]:
train_dataset = FashionMNIST(
    root="./data", train=True, download=True, transform=ToTensor()
)
test_dataset = FashionMNIST(
    root="./data", train=False, download=True, transform=ToTensor()
)

In [47]:
X_train = train_dataset.data.float()
y_train = train_dataset.targets
X_test = test_dataset.data.float()
y_test = test_dataset.targets

In [48]:
@dataclass
class TrainConfig:
    lr: float = 1e-3
    total_iterations: int = 100


# Для оценки будем использовать метрику accuracy
# Подумайте (опционально), какие еще метрики можно использовать
def calculate_accuracy(y_pred: torch.Tensor, y_true: torch.Tensor) -> float:
    _, predicted = torch.max(y_pred, 1)
    correct = (predicted == y_true).float().sum()
    accuracy = correct / y_true.shape[0]
    return accuracy.item()

## Задание №1

Попробуйте реализовать простой бейзлайн с несколькими слоями:
- Linear
- ReLU
- Linear

Организуйте слои в nn.Sequential переменную с названием net (т.е. ваш класс должен иметь атрибут self.net). Вам нужно дописать и сдать как `SimpleModel`, так и `train_loop`.
Используйте кросс-энтропию как функцию потерь.

In [86]:
import tqdm

class SimpleModel(nn.Module):

    def __init__(self, num_classes: int = 10):
        """
        Args:
            num_classes: Количество классов для классификации
        """
        super().__init__()
        # Размерность скрытого слоя
        hidden_dim = 512

        self.net = nn.Sequential(
            nn.Linear(in_features=28 * 28, out_features=hidden_dim),
            nn.ReLU(),
            nn.Linear(in_features=hidden_dim, out_features=num_classes)
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = x.reshape((-1, 28 * 28))
        x = self.net(x)
        return x


def train_loop(
    model: SimpleModel,
    X_train: torch.Tensor,
    y_train: torch.Tensor,
    X_val: torch.Tensor,
    y_val: torch.Tensor,
    config: TrainConfig,
):
    """Обучите здесь модель, подсчитайте метрики на валидационной выборке.

    Можете так же писать/рисовать accuracy в процессе обучения.
    Например, каждые 10 итераций или даже каждую итерацию.
    """
    # Оставьте такое название перменной, это требование проверяющей системы
    optimizer = optim.SGD(model.parameters(), config.lr)
    # Ваш код обучения модели

    for i in tqdm.trange(config.total_iterations):
        optimizer.zero_grad()

        output = model(X_train)
        loss = F.cross_entropy(output, y_train)
        loss.backward()
        optimizer.step()

    with torch.no_grad():
        model.eval()
        loss_val = calculate_accuracy(model(X_val), y_val)	
        model.train()
        return loss_val


## Задание №2
Какое максимальное значение метрики accuracy удалось получить в процессе обучения на **тестовой** выборке?

Округлите до 3 значений после запятой


In [62]:
torch.manual_seed(987)
# дефолтные значения
config = TrainConfig()
# Ваш код для обучения и подсчета accuracy
model = SimpleModel()
train_loop(model, X_train, y_train, X_test, y_test, config)

100%|██████████| 100/100 [01:16<00:00,  1.31it/s]


0.7634999752044678

## Задание №3
Добавьте один `dropout` слой в вашу модель.

_Подумайте, что может поменяться при перестановке ReLU и Dropout слоев местами._

In [63]:
class DropoutModel(nn.Module):

    def __init__(self, num_classes=10, dropout_rate=0.2):
        """
        Args:
            num_classes: Количество классов для классификации
            dropout_rate: Вероятность отключения нейронов
        """
        super().__init__()
        # Размерность скрытого слоя
        hidden_dim = 512
        self.net = nn.Sequential(
            nn.Linear(in_features=28 * 28, out_features=hidden_dim),
            nn.ReLU(),
            nn.Dropout(p=dropout_rate),
            nn.Linear(in_features=hidden_dim, out_features=num_classes)
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = x.reshape((-1, 28 * 28))
        x = self.net(x)
        return x

In [64]:
torch.manual_seed(987)
config = TrainConfig()
# Ваш код для обучения и подсчета accuracy
model = DropoutModel()
train_loop(model, X_train, y_train, X_test, y_test, config)

  0%|          | 0/100 [00:00<?, ?it/s]

100%|██████████| 100/100 [01:37<00:00,  1.02it/s]


0.7803999781608582

## Задание №4
Какое максимальное значение accuracy получилось в ходе обучения модели на **тестовой** выборке?

Округлите до 3х знаков после запятой и отправьте в ЛМС.

## Задание №5

Добавьте `BatchNorm` в вашу модель.
Отправьте в ЛМС реализацию.

Стоит ли делать BatchNorm до ReLU или после него?
Это дискуссионный вопрос, чаще всего применяют сначала нелинейность, затем Batch Norm.
Один из аргументов: при таком подходе данные на выходе будут иметь среднее 0 - что и ожидают люди, когда добавляют нормализацию.

_[Дискуссия на Reddit](https://www.reddit.com/r/MachineLearning/comments/67gonq/d_batch_normalization_before_or_after_relu/)_

Для определенности в этом задании будем следовать такому порядку: сначала ReLU, затем Batch Norm.

In [51]:
class BatchNormModel(nn.Module):

    def __init__(self, num_classes=10):
        """
        Args:
            num_classes: Количество классов для классификации
        """
        super().__init__()
        # Размерность скрытого слоя
        hidden_dim = 512
        self.net = nn.Sequential(
            nn.Linear(in_features=28 * 28, out_features=hidden_dim),
            nn.ReLU(),
            nn.BatchNorm1d(num_features=hidden_dim),
            nn.Linear(in_features=hidden_dim, out_features=num_classes)
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = x.reshape((-1, 28 * 28))
        x = self.net(x)
        return x

In [83]:
torch.manual_seed(987)
config = TrainConfig()
# Ваш код для обучения и подсчета accuracy
model = BatchNormModel()
train_loop(model, X_train, y_train, X_test, y_test, config)

100%|██████████| 100/100 [00:14<00:00,  6.73it/s]


0.7039999961853027

## Задание №6
Какое максимальное значение `accuracy` получилось в ходе обучения модели на **тестовой** выборке?

Округлите до 3х знаков после запятой.

Результат batch normalization мог не особо порадовать.
Но не спешите с выводами насчет этого слоя!

Попробуйте обучить заново все три модели со значением `lr=1e-2` (в 10 раз больше).
Сравните результаты моделей и сделайте вывод.

In [85]:
torch.manual_seed(987)
new_config = TrainConfig(lr=1e-2)
# Ваш код для обучения и подсчета accuracy
model = BatchNormModel()
train_loop(model, X_train, y_train, X_test, y_test, new_config)

0.8119999766349792

## Задание №7
Добавьте `LRscheduler` в вашу модель.

Подробнее про `schedulers` можно почитать в [документации](https://pytorch.org/docs/stable/optim.html#how-to-adjust-learning-rate)

In [82]:
from torch.optim.lr_scheduler import StepLR


def train_loop_with_scheduler(
    model,
    X_train: torch.Tensor,
    y_train: torch.Tensor,
    X_val: torch.Tensor,
    y_val: torch.Tensor,
    config: TrainConfig,
):
    optimizer = optim.SGD(model.parameters(), config.lr)
    scheduler = StepLR(optimizer, step_size=5, gamma=0.1)

    for i in tqdm.trange(config.total_iterations):
        optimizer.zero_grad()

        output = model(X_train)
        loss = F.cross_entropy(output, y_train)
        loss.backward()
        optimizer.step()
        scheduler.step()
    
    with torch.no_grad():
        model.eval()
        loss_val = calculate_accuracy(model(X_val), y_val)	
        model.train()
        return loss_val

In [69]:
torch.manual_seed(987)
config = TrainConfig(lr=1e-3)
# Ваш код для обучения и подсчета accuracy
model = BatchNormModel()
train_loop_with_scheduler(model, X_train, y_train, X_test, y_test, config)

100%|██████████| 100/100 [01:18<00:00,  1.27it/s]


0.17900000512599945

## Задание №8

Поэксперементируйте с параметрами нейронной сети, попробуйте добиться максимальной метрики `accuracy`.

- попробуйте комбинацию Dropout + Batch Normalization и подумайте, как лучше всего раскрыть силу Batch Normalization (вспомните эксперименты с lr);
- попробуйте подвигать вероятность в Dropout;
- ну, или подержите обучение подольше, поставив больше шагов :)

В ЛМС нужно сдать код класса `ExpModel`.
Вам необходимо выбить accuracy > 80%, чтобы сдать этот пункт.

In [79]:
torch.manual_seed(987)


# Ваш код модели и ее обучения при seed = 987
class ExpModel(nn.Module):

    def __init__(self, num_classes=10):
        """
        Args:
            num_classes: Количество классов для классификации
        """
        super().__init__()
        hidden_dim = 512
        self.net = nn.Sequential(
            nn.Linear(in_features=28 * 28, out_features=hidden_dim),
            nn.ReLU(),
            nn.Dropout(p=0.3),
            nn.BatchNorm1d(num_features=hidden_dim),
            nn.Linear(in_features=hidden_dim, out_features=num_classes)
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = x.reshape((-1, 28 * 28))
        x = self.net(x)
        return x


model = ExpModel()
config = TrainConfig(lr=10e-2, total_iterations=200)
train_loop(model, X_train, y_train, X_test, y_test, config)

100%|██████████| 200/200 [00:58<00:00,  3.40it/s]


0.8636999726295471

Наконец, сохраним лучшую модель, чтобы в будущем ее могли взять и использовать, без обучения.

## Задание №9

Напишите код, который сохранит модель в файл `model.pt`.

In [80]:
torch.save(model, "model.pt")

In [81]:
# Впоследствии эту модель можно будет загрузить вот так:
model_loaded = ExpModel(num_classes=len(y_test.unique()))
model_loaded.load_state_dict(torch.load("model.pt"))

UnpicklingError: Weights only load failed. This file can still be loaded, to do so you have two options, [1mdo those steps only if you trust the source of the checkpoint[0m. 
	(1) In PyTorch 2.6, we changed the default value of the `weights_only` argument in `torch.load` from `False` to `True`. Re-running `torch.load` with `weights_only` set to `False` will likely succeed, but it can result in arbitrary code execution. Do it only if you got the file from a trusted source.
	(2) Alternatively, to load with `weights_only=True` please check the recommended steps in the following error message.
	WeightsUnpickler error: Unsupported global: GLOBAL __main__.ExpModel was not an allowed global by default. Please use `torch.serialization.add_safe_globals([__main__.ExpModel])` or the `torch.serialization.safe_globals([__main__.ExpModel])` context manager to allowlist this global if you trust this class/function.

Check the documentation of torch.load to learn more about types accepted by default